# 見かけ上無関係な回帰による住宅エネルギー需要システムの推定

## エグゼクティブサマリー

ある地域の公益事業者が、2 式からなる住宅 **エネルギー需要システム**――電気とガスの予算シェアを、自価格・交差価格・世帯所得の関数として――を、見かけ上無関係な回帰（SUR）法による **PROC SYSLIN** で推定する。2 つのシェア式は共通の世帯予算を共有するため、その誤差は交差相関する。SUR はその相関を活用するために式を同時推定し、符号の整合した自価格・交差価格効果を復元し、需要分析者が必要とする式間共分散と制約検定を提供する。

## データソース

| データセット | 行数 | 粒度 | 主な変数 | 説明 |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | 月次市場観測ごとに 1 行 | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | 合成の月次住宅エネルギー市場パネル。`lp_elec`／`lp_gas` は電気とガスの対数実質価格、`lincome` は対数実質世帯所得、`w_elec`／`w_gas` は電気とガスの支出予算シェアで、既知の AIDS 型需要構造に相関のある式間ノイズを加えて生成した。 |

# 見かけ上無関係な回帰による住宅エネルギー需要システムの推定

ある地域のガス・電気公益事業者は、相対価格と所得が変化するにつれて、世帯がエネルギー予算を **電気** と **ガス** の間でどう再配分するかを理解したい。自然な枠組みは **需要システム**、すなわち同時推定される予算シェア式の集合である。

2 つの特徴が同時推定を適切な手法にする：

- シェア式は共通の世帯予算に依拠するため、その誤差は **交差相関** する――世帯が電気により多く支出すればガスへの支出は減る。式を **見かけ上無関係な回帰（SUR）** でまとめて推定することで、その相関を効率のために活用する。
- 経済理論は、システム推定量が直接課すことのできる **線形制約**（加算性、同次性、対称性）を課す。

`SUR` オプション付きの `PROC SYSLIN` は、まさにこの状況のために作られている。各シェア式を当てはめ、式間の誤差共分散を推定し、その共分散でシステムを再推定して、効率的で理論整合的な弾力性を――式間共分散行列と同時制約検定とともに――提供する。

このノートブックでは：

1. 既知のシェア構造から合成の月次エネルギー市場パネルを生成する。
2. 2 式のシェアシステムを SUR で当てはめる。
3. 同時 SUR フィットを式ごとの OLS と比較する。
4. 同次性制約を課し、その同時 F 検定を読む。
5. 弾力性報告のために係数表を抽出する。

## ステップ 1 — 合成エネルギー市場パネルの生成

小さな **2 財エネルギー需要システム**（電気とガス）の 60 個の月次観測をシミュレートする。データ生成過程は線形化された AIDS 型の予算シェアモデルである：

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

意図的に **式間相関** を組み込む：世帯が電気により多く支出するとガスへの支出は減るため、`e1` と `e2` は負に相関する。共通のエネルギー市場価格因子も両方の対数価格を同時に動かす。これらが、各式を単独で当てはめるよりも同時 SUR 推定を有利にする特徴である。

In [1]:
データ energy;
   呼出 streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   繰返 month = 1 から 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      出力;
   終了;

   見出 w_elec="電力支出シェア" w_gas="ガス支出シェア"
        lp_elec="対数電力価格" lp_gas="対数ガス価格" lincome="対数所得";
   保持 month lp_elec lp_gas lincome w_elec w_gas;
実行;

処理 平均 データ=energy n mean std MIN MAX maxdec=3;
   変数 w_elec w_gas lp_elec lp_gas lincome;
   表題 "エネルギー支出シェアと価格・所得の記述統計";
実行;
表題;


                                                 エネルギー支出シェアと価格・所得の記述統計                                                  

                                                  The MEANS Procedure

 Variable  Label                        N           Mean     Std Dev     Minimum     Maximum
 -------------------------------------------------------------------------------------------
 w_elec    電力支出シェア                     60          0.440       0.011       0.417       0.464
 w_gas     ガス支出シェア                     60          0.228       0.014       0.198       0.256
 lp_elec   対数電力価格                      60          4.617       0.142       4.354       4.911
 lp_gas    対数ガス価格                      60          4.211       0.162       3.818       4.566
 lincome   対数所得                        60         10.916       0.088      10.723      11.174
 -------------------------------------------------------------------------------------------




NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## ステップ 2 — 需要システムのベースライン SUR 推定

両方のシェア式を同時推定する。`SUR` オプションは、式間の誤差共分散を推定し、それを効率的な実行可能 GLS フィットに用いるよう `PROC SYSLIN` に指示する。2 つのラベル付き `MODEL` ステートメント（`elec` と `gas`）がシステムを定義し、それぞれ予算シェアを 2 つの対数価格と対数所得に回帰する。

SYSLIN は各式のパラメータ推定値を報告し、最後に **式間共分散行列（Cross-Model Covariance Matrix）**――SUR が活用する 2 式の誤差の推定共分散――を報告する。

In [2]:
処理 syslin データ=energy sur;
   elec: 模型 w_elec = lp_elec lp_gas lincome;
   gas:  模型 w_gas  = lp_elec lp_gas lincome;
実行;



                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: 電力支出シェア

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: ガス支出シェア

  Number of Observations                       60
  SSE                                     


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## ステップ 3 — 式ごとの OLS との比較

SUR の利点を見るため、同じ 2 式を通常最小二乗（既定の手法、1 式ずつ）で再推定する。OLS は式間の誤差相関を無視するため、一致性はあるが、その相関がゼロでないとき――ここでは構成上そうである――SUR より効率が劣る。

2 つの係数表を比較すると、システム構造を考慮したときに推定値とその標準誤差がどう変化するかが分かる。

In [3]:
処理 syslin データ=energy ols;
   elec: 模型 w_elec = lp_elec lp_gas lincome;
   gas:  模型 w_gas  = lp_elec lp_gas lincome;
実行;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: 電力支出シェア

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: ガス支出シェア

  Number of Observations                       60
  SSE                                     


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## ステップ 4 — 経済理論を課して検定する

需要理論は、名目上の価格／所得効果が **0 次同次性** に従うべきだと述べる：すべての価格と所得をスケールしても実質需要は変わらないため、1 つの式内の価格係数と所得係数の和はゼロになる。これを `RESTRICT` ステートメントで電気シェアに課す。

SYSLIN は制約に従ってシステムを再推定し、制約がデータと整合するかの **制約結果（Restriction Results）** F 検定を報告する――同次性仮説に対する直接的な同時検定である。

In [4]:
処理 syslin データ=energy sur;
   elec: 模型 w_elec = lp_elec lp_gas lincome;
   gas:  模型 w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
実行;



                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: 電力支出シェア

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: ガス支出シェア

  Number of Observations                       60
  SSE                                     


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## ステップ 5 — 弾力性報告のための推定値の取得

最終報告のため、収束した係数を `OUTEST=` で保存する。得られるデータセットは式ごとに 1 行を持ち、切片と傾きの推定値に加えて適合統計量を含み、公益事業者の自価格・交差価格弾力性計算を標本平均シェアで支える。記録のためにそれを印字する。

In [5]:
処理 syslin データ=energy sur outest=demand_est;
   elec: 模型 w_elec = lp_elec lp_gas lincome;
   gas:  模型 w_gas  = lp_elec lp_gas lincome;
実行;

処理 印刷 データ=demand_est noobs 見出;
   見出 lp_elec="対数電力価格" lp_gas="対数ガス価格" lincome="対数所得";
   表題 "SUR 需要システムの係数推定値";
実行;
表題;



                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: 電力支出シェア

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: ガス支出シェア

  Number of Observations                       60
  SSE                                     


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## 結果の解釈

**符号の整合性。** SUR 推定はデータに組み込まれた代替構造を復元する。電気シェア式では **自対数価格係数が正**（`LP_ELEC` = 0.112）で、**交差価格係数は負**（`LP_GAS` = -0.098）である。ガスシェア式ではそのパターンが鏡映する（自 `LP_GAS` = 0.114、交差 `LP_ELEC` = -0.075）。すべての自価格・交差価格効果は統計的に有意（各 `Pr > |t|` は 0.005 未満）であり、代替の符号はノイズの多いフィットの産物ではなく確固として同定されている。

**SUR は式間相関を活用する。** **式間共分散行列（Cross-Model Covariance Matrix）** は負の非対角要素（-0.000068）を示す：世帯は電気支出をガス支出と天秤にかける――まさに構成どおりである。その共分散がゼロでないため、同時 SUR 推定は各式を単独で当てはめるより効率的である――ステップ 2 の SUR 標準誤差は、ステップ 3 の式ごとの OLS 標準誤差より一様にわずかに小さい（たとえばガスの `LP_ELEC` 標準誤差は OLS の 0.0264 から SUR の 0.0255 へ下がる）一方で、点推定値は変わらない。そのより締まった推測が、2 つの別々の回帰ではなくシステムをモデル化する見返りである。

**理論は成立する。** `RESTRICT` を介して電気シェアに **0 次同次性**（価格係数と所得係数の和がゼロ）を課したところ、制約結果 F 検定は 2.14、`Pr > F` = 0.149 となった。制約は **棄却されない** ため、消費者需要理論の同次性の予測はこの標本と整合する――公益事業者は制約付きの理論整合的な推定値を自信をもって報告へ持ち込める。

**運用上の利用。** `OUTEST=` 表は式ごとに 1 行、切片と傾きの推定値および適合統計量を保存する。これらの係数は、標本平均シェア（ステップ 1 より `W_ELEC` ~ 0.44、`W_GAS` ~ 0.23）における自価格・交差価格弾力性と所得（支出）弾力性へ直接変換され、需要予測と料金設計シナリオのための、擁護可能で理論整合的な基礎を公益事業者に与える。